# DQN 原理笔记
DQN与Q-learning最大的区别：引入经验回放池（防止连续动作下训练网络造成的过拟合），2个网络

## 目录

| 章节 | 内容 |
|---|---|
| 一 | 原理一：从增量更新Q表到梯度下降
| 二 | 原理二：为什么当前网络要逼近"从自己复制来的"目标网络 |
| 三 | 改进一：Double DQN —— 消除 Q 值高估 |
| 四 | 改进二：Dueling DQN —— 解耦Q |
| 五 | 改进三：N-step Learning —— 目标值看得更远 |
| 六 | 改进四：Noisy Net —— 参数噪声代替 ε 探索 |
| 七 | 改进五：Prioritized Replay —— 按误差加权抽样 + 存档续训 |

## 一、原理一：从增量更新Q表到梯度下降

### 1.1 先回忆 Q-learning 的增量更新

你学过的 Q-learning 更新公式是：

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[\underbrace{r + \gamma\max_{a'}Q(s',a')}_{\text{目标值 target}} - Q(s,a)\Big]$$

它的意思是：把当前的 $Q(s,a)$ 往"目标值 $\text{target}$"方向拉近一点，拉近的幅度由学习率 $\alpha$ 控制。

### 1.2 关键推导：增量更新就是"对平方损失做一步梯度下降"

定义单个样本的平方误差损失（取 $\frac{1}{2}$ 只是为了求导方便，不影响最优解）：

$$L = \frac{1}{2}\big(Q - \text{target}\big)^2$$

对 $Q$ 求梯度：

$$\frac{\partial L}{\partial Q} = Q - \text{target}$$

用学习率 $\alpha$ 做一步梯度下降：

$$Q \leftarrow Q - \alpha \cdot \frac{\partial L}{\partial Q} = Q - \alpha(Q - \text{target}) = Q + \alpha(\text{target} - Q)$$

**右边的式子就是 Q-learning 的增量更新公式！**

> **结论：对平方误差损失做一步梯度下降 ⟺ 做一步 Q-learning 增量更新。二者是同一个数学操作，只是"包装"不同。**



既然两者数学上等价，为什么代码里写法差别这么大？答案在于**参数是否共享**：

| | 表格 Q-learning | DQN（神经网络） |
|---|---|---|
| 被更新的对象 | Q 表里**一个独立的格子** $Q(s,a)$ | 网络**全部权重** $\theta$ |
| 更新 $Q(s,a)$ 会影响 $Q(s',a')$ 吗 | **不会**（各格子互相独立） | **会**（所有状态共用同一组 $\theta$） |
| 适合的写法 | 单个标量直接加减 | 显式写出损失函数，对 $\theta$ 求梯度 |

### 1.3 表格法：一格一格改

表格法里每个 $Q(s,a)$ 是**独立参数**，改一格不影响其他格。所以可以"哪格错改哪格"，直接写增量式 $Q \leftarrow Q + \alpha(\text{target} - Q)$。这一步本质上**就是**对该格子的平方损失做一步梯度下降，只是被"简化"成了标量加减。

### 1.4 神经网络：参数共享，只能整体优化

神经网络的权重 $\theta$ 被**所有状态共用**——你没法"只改状态 $s$ 对应的那部分网络"。所以必须把目标写成一个**优化问题**：

$$\theta^{*} = \arg\min_{\theta} \; \mathbb{E}_{(s,a,r,s') \sim \text{回放池}}\Big[\big(Q(s,a;\theta) - \text{target}\big)^2\Big]$$

然后对整个 $\theta$ 做梯度下降。

### 1.5 批量的"期望"（1/B Σ）是怎么回事？

DQN 一次从经验池抽 $B$ 条经验（mini-batch），损失取平均：

$$\text{loss} = \frac{1}{B}\sum_{i=1}^{B}\Big(Q(s_i,a_i;\theta) - \text{target}_i\Big)^2$$

这是在用**样本平均**近似"回放池分布下的期望损失"（蒙特卡洛估计）。梯度也取平均再更新（SGD），好处是：

- 单条经验噪声大、可能带偏方向；**平均梯度**更接近真实梯度方向，更新更稳；
- 打平了相邻经验之间的相关性（配合经验回放的随机抽样）。

> **一句话**：增量更新和平方损失是同一个数学操作（梯度下降最小化 TD 误差的平方）；表格法参数独立所以能"一格一格改"，网络法参数共享所以必须写成"批量损失 + 梯度下降"。

## 二、原理二：为什么当前网络要"逼近"从自己复制来的目标网络？

### 2.1 先看清时间线：更新 vs 复制

DQN 训练中有两个结构完全相同的网络：

- `q_net`（当前网络）：**每一步**都更新，参数时刻在变；
- `target_net`（目标网络）：**每 10 回合**才从 `q_net` 复制一次参数，其余时间**完全冻结**。

所以在两次复制之间，`target_net` 就是 `q_net` 在 **10 回合前的一张快照**。当前网络每步更新的内容，就是让自己这张"照片"中的 Q 值估计，更接近这张 10 回合前的快照算出来的目标值。

### 2.2 如果没有目标网络会怎样？（移动靶问题）

假设 target 直接用当前网络自己算：$\text{target} = r + \gamma\max_{a'}Q(s',a';\theta)$，而 $\theta$ 正是正在被更新的参数。

问题出在**参数共享**上：$Q(s,a)$ 和 $Q(s',a')$ 共用同一组权重 $\theta$。梯度下降把 $Q(s,a)$ 往上推的同时，也会把 $Q(s',a')$ 往上推——而 $Q(s',a')$ 又是下一个 target 的组成部分。于是形成正反馈：

$$Q(s,a)\uparrow \;\Rightarrow\; \text{target}\uparrow \;\Rightarrow\; Q(s,a)\text{ 被要求更高}\;\Rightarrow\; \cdots$$

结果 Q 值被自己不断抬升，越学越大、最终发散（这就是著名的"自举发散 / 过度估计"问题）。直观地说：**靶子跟着你的准星一起动，你永远瞄不准。**

> 注意：代码里 `with torch.no_grad()` 只能保证"单步内" target 不参与求导，挡不住"跨步"的问题——下一步 target 仍会因 $\theta$ 变了而变。所以 `no_grad` 解决不了移动靶，**独立的目标网络**才是解决方案。

### 2.3 表格法为什么没有这个问题？

因为在表格法里，$Q(s,a)$ 和 $Q(s',a')$ 是**两个不同的格子**——更新 $Q(s,a)$ 那格，$Q(s',a')$ 那格纹丝不动。所以即使"用自己当前的表算 target"，对每个格子而言 target 实际上是不动的。**格子间的独立性天然切断了正反馈回路。**

神经网络恰恰**破坏了这种独立性**（参数共享），所以必须人为造一个"不动的表"——那就是 `target_net`。

### 2.4 那为什么还要定期"刷新"目标网络？

因为**静止的靶子虽然好瞄，但可能瞄偏了**。10 回合前的快照算出来的 target 可能不准确（网络还太年轻）。刷新一下，把靶子挪到"当前对 $Q^{*}$ 更好的估计"的位置上，再继续瞄。

这个过程本质是**不动点迭代**——逐步逼近贝尔曼方程 $Q = r + \gamma\max Q'$ 的解 $Q^{*}$：先固定右边、解出左边的 $Q$，再把解代回右边……反复迭代直到收敛。

## 三、改进一：Double DQN —— 消除 Q 值高估

### 3.1 问题：DQN 的目标值会系统性高估

基准 DQN 的目标值用 $\max$ 取"下一状态的最大 Q 值"：

$$\text{target} = r + \gamma\max_{a'}Q(s',a';\theta^-)$$

问题是：$Q(s',a')$ 是**带估计误差**的（神经网络不完美，估计值 = 真值 + 噪声）。对一组带噪声的估计取最大值，结果**只会偏大、不会偏小**——因为 $\max$ 恰好挑中那个被噪声抬得最高的动作。数学上就是琴生不等式：

$$\mathbb{E}\big[\max(\hat{Q})\big] \;\ge\; \max\big(\mathbb{E}[\hat{Q}]\big)$$

更糟的是，基准 DQN 里**选动作和评价值用的是同一个网络**（`target_net`）：同一个有偏估计器"又当运动员又当裁判"，高估会自我放大，最终可能把"虚高的次优动作"当最优，学到次优策略。

### 3.2 解法：把"选动作"和"评价值"解耦

DDQN 的核心只有一句话：

- **基准 DQN**：$a^* = \arg\max_{a'}Q(s',a';\theta^-)$，再取 $Q(s',a^*;\theta^-)$ —— **同一个网络**又选又评；
- **DDQN**：$a^* = \arg\max_{a'}Q(s',a';\theta)$（**当前网络**选），再取 $Q(s',a^*;\theta^-)$（**目标网络**评）。

写成公式：

$$\text{target}_{\text{DDQN}} = r + \gamma\, Q\!\big(s',\; \underbrace{\arg\max_{a'}Q(s',a';\theta)}_{\text{当前网络选动作}},\; \theta^-\big)$$

### 3.3 为什么可行（数学原理）

高估的根源是"**同一个有偏估计器既选又评**"。DDQN 用两组参数（$\theta$ 与 $\theta^-$）分别负责选择和评价——要两个网络**同时**对同一个动作高估，偏差才会放大，而这种情况的概率远小于"一个网络自己高估自己"。van Hasselt (2016) 证明了双估计器的期望值**不超过**单估计器，即系统性偏差被压了下去。DDQN 不是"绝对不高估"，而是把系统性高估显著压低（有时甚至会略低估）。假设当前网络因为噪声，对某个非最优动作( a_bad) 产生了一个虚高的误判，导致它被当前网络错误地选中。如果是传统DQN，这个虚高的分数直接就被采纳了，但在 DDQN 中，这个被选中的动作 ( a_bad)  会被送进目标网络（参数不同、噪声独立）。目标网络对 ( a_bad)  的评估通常是比较客观或者偏向真实均值的，不会带着当前网络盲目乐观的滤镜。因此，虚高的误差无法在同一步计算中自我实现并叠加。

### 3.4 深入：为什么"当前网络选、目标网络评"就不会高估？

把 Q 值估计写成"真值 + 噪声"：

$$\hat Q(s',a) = Q^*(s',a) + \varepsilon(s',a)$$

其中 $\varepsilon$ 是均值约为 0 的估计误差。标准 DQN 取 $\max_a \hat Q(s',a)$，问题在于：被选中的 $a^*$ 往往正是因为它**恰好噪声偏大、为正**才被选中，于是"选动作时的好运气（正噪声）"被原封不动地带进了评估值——这就是系统性高估的来源：

$$\mathbb{E}\big[\max_a \hat Q(s',a)\big] = \max_a Q^*(s',a) + \underbrace{\mathbb{E}[\varepsilon \mid \text{被选中}]}_{>\,0}$$

DDQN 则把"选中时的噪声"和"评分时的噪声"分开：

$$a^* = \arg\max_a \underbrace{Q^*(s',a) + \varepsilon_1(s',a)}_{\text{当前网络 } \theta \text{ 的估计}}$$

$$\text{评估值} = Q^*(s',a^*) + \underbrace{\varepsilon_2(s',a^*)}_{\text{目标网络 } \theta^- \text{ 的估计}}$$

$a^*$ 是被 $\varepsilon_1$（当前网络的噪声）抬起来的，但评分用的是 $\varepsilon_2$（目标网络的噪声）。因为 $\varepsilon_2$ 与"$\varepsilon_1$ 恰好把 $a^*$ 抬高"这件事无关，所以在期望上：

$$\mathbb{E}\big[\varepsilon_2(s',a^*) \mid a^* \text{ 被 } \varepsilon_1 \text{ 选中}\big] = \mathbb{E}[\varepsilon_2(s',a^*)] = 0$$

**"选中的那份运气"不再被计入评分**——评估值在期望上回到真值附近，系统性偏差被抹掉。一句话：**标准 DQN 是"谁选的谁打分"，选中的正噪声被原样记分；DDQN 是"A 选人、B 打分"，A 的运气不会带进 B 的成绩单。**

> 补充（实际代码中的效果不是时刻满格）：$\theta$ 和 $\theta^-$ 每 10 回合会同步一次，刚同步完两者完全相同，DDQN 暂时退化成 DQN（高估恢复）；之后 `q_net` 每步更新、与目标网络越走越远，两套噪声逐渐去相关，降偏效果才逐步显现。

### 3.5 代码上只改一处

整个算法只在 `update()` 的目标值计算里换了一行：

| | 代码 |
|---|---|
| 基准 DQN | `max_next_q = target_net(next_states).max(dim=1, keepdim=True)[0]` |
| DDQN | `best_actions = q_net(next_states).argmax(dim=1, keepdim=True)`（当前网络选） |
| | `max_next_q = target_net(next_states).gather(1, best_actions)`（目标网络评） |

网络结构、经验回放池、ε 贪婪、目标网络同步机制、训练循环**全部不变**。

> 这里正好呼应基准代码里的两个操作：`argmax` 要的是"最优动作的下标"，`gather` 要的是"该下标对应的 Q 值"——一个负责"选"，一个负责"评"。

## 改进二：Dueling DQN原理，解耦Q

### 1.1 前向传播（定义计算图）

设输入特征矩阵为 $ X $（形状 $ B \times D $），参数为 $ W_v $（$ D \times 1 $）和 $ W_a $（$ D \times N $）。

1.  **计算状态价值**：
    $$ V = X W_v \quad (B \times 1) $$

2.  **计算原始优势**：
    $$ A = X W_a \quad (B \times N) $$

    X内部包含  了若干线性层和非线性层，它是两条支路共享的底层特征提取器。

3.  **中心化优势（减去行均值）**：
    这一步是为了解决不可辨识性。设 $ \bar{A} = \frac{1}{N} \sum_{a=1}^N A_{i,a} $（对每个样本的动作求均值），则：
    $$ A_{centered} = A - \bar{A} \quad (B \times N) $$
    网络直接输出的原始A 只是一个可学习的无约束底数（bias），它的绝对值没有物理意义，只有减去均值之后的那部分才代表优势。
4.  **合成 Q 值**：
    Q 是 V 与中心化优势的**逐元素相加**（V 通过广播复制 N 份）：
    $$ Q = V + A_{centered} \quad (B \times N) $$

5.  **计算损失 Loss**（标量）：
    使用均方误差，其中 $ Q_{target} $ 是目标网络给出的固定值（不参与求导）：
    $$ \mathcal{L} = \frac{1}{B \times N} \sum_{i=1}^{B} \sum_{a=1}^{N} (Q_{i,a} - Q^{target}_{i,a})^2 $$


### 1.2 反向传播起点（Loss 对 Q 求偏导）

虽然 $ \mathcal{L} $ 是标量，但它对 $ Q $ 的每一个元素都有偏导数。对上面的平方损失求导（幂函数求导），得到：

$$ \frac{\partial \mathcal{L}}{\partial Q_{i,a}} = \frac{2}{B \times N} (Q_{i,a} - Q^{target}_{i,a}) $$

记这个梯度矩阵为 $ dQ $，形状 $ B \times N $。这里的数值大小取决于每个动作自己的预测误差。

dQ就是128行（样本数）2列（动作数）的矩阵，每一行值为当前网络在这128个样本中的某个状态a输入后得到的2个q值-目标网络输入状态a得到的2个q值乘以2/(128*2)。就是乘2（来自于求导时的平方）除以样本数乘动作数


### 1.3 梯度经过“加法节点”（$ Q = V + A_{centered} $）

在反向传播中，加法节点的规则是：**上游梯度直接原样传递给每一个加数**。

比如3+4=7，3和4的变化量对于7的影响是一样的，也就是说加法节点的输出误差梯度就是原样复制给每个加数


梯度就是输出对某个输入的导数，衡量这个输入变换量对输出的影响。

在乘法节点，一个乘数收到的梯度，等于上游总误差乘以“另一个乘数”的当前前向值。

就是比如z=xy，z对x求偏导，是y，z对y求偏导是x，也就是说x的变化量对于z的影响取决于y，同理。

为什么？因为 $ Q $ 对 $ V $ 的偏导为 1，且 $ Q $ 对 $ A_{centered} $ 的偏导也为 1。根据链式法则，相乘后梯度不变。

所以在这一步，两份梯度完全相等：
$$ \frac{\partial \mathcal{L}}{\partial V} = dQ \quad (B \times 1) $$
$$ \frac{\partial \mathcal{L}}{\partial A_{centered}} = dQ \quad (B \times N) $$


### 1.4 梯度经过“中心化节点”（$ A_{centered} = A - \bar{A} $）

这是理解 Dueling DQN 的关键。前向时，每个动作的 $ A_{centered} $ 不仅取决于自身的 $ A $，还通过均值 $ \bar{A} $ 取决于其他所有动作的 $ A $。因此在反向传播时，我们不能只看自己，必须考虑均值对每个动作的耦合影响。

**（1）先看单个原始优势 $ A_{i,a} $ 是如何影响所有输出 $ A_{centered_{i,k}} $ 的**

中心化的前向数学定义是：
$$ A_{centered_{i,k}} = A_{i,k} - \bar{A}_i $$
而均值 $ \bar{A}_i = \frac{1}{N} (A_{i,1} + A_{i,2} + \dots + A_{i,a} + \dots + A_{i,N}) $。

为了反向传播，我们需要计算偏导数：$ \frac{\partial A_{centered_{i,k}}}{\partial A_{i,a}} $，即当原始的 $ A_{i,a} $ 变化时，中心化后的第 $ k $ 个输出会变多少。

代入上式并对 $ A_{i,a} $ 求导，得到两个系数：

- **当 $ k = a $ 时（自己的变化对自己的输出影响）**：
  $$ \frac{\partial A_{centered_{i,a}}}{\partial A_{i,a}} = 1 - \frac{1}{N} $$
  因为 $ A_{i,a} $ 直接贡献给自身输出（系数1），同时又通过均值贡献给自身输出（系数 $ -\frac{1}{N} $）。

- **当 $ k \neq a $ 时（自己的变化对其他动作的输出影响）**：
  $$ \frac{\partial A_{centered_{i,k}}}{\partial A_{i,a}} = 0 - \frac{1}{N} = -\frac{1}{N} $$
  因为 $ A_{i,a} $ 不直接出现在第 $ k $ 个输出中（系数0），但它通过均值 $ \bar{A} $ 出现在第 $ k $ 个输出中（系数 $ -\frac{1}{N} $）。

**（2）再看链式法则如何把上游梯度 $ dQ $ 传给 $ A $**

根据链式法则，$ \frac{\partial \mathcal{L}}{\partial A_{i,a}} $ 需要把 $ A_{i,a} $ 对所有输出 $ A_{centered_{i,k}} $（即 $ k=1 $ 到 $ N $）的影响全部加起来。所以我们需要对所有 $ k $ 进行求和：

$$ \frac{\partial \mathcal{L}}{\partial A_{i,a}} = \sum_{k=1}^{N} \frac{\partial \mathcal{L}}{\partial A_{centered_{i,k}}} \cdot \frac{\partial A_{centered_{i,k}}}{\partial A_{i,a}} $$

把上游梯度记作 $ dQ_{i,k} = \frac{\partial \mathcal{L}}{\partial A_{centered_{i,k}}} $，并把上面求出的两个系数代入。为了清晰地看到“代入并整理”的过程，我们把这个求和拆成两项：一项是 $ k=a $（自己），另一项是 $ k \neq a $（其他所有动作）。

$$ dA_{i,a} = \left( dQ_{i,a} \times (1 - \frac{1}{N}) \right) + \sum_{k \neq a} \left( dQ_{i,k} \times (-\frac{1}{N}) \right) $$

展开第一项，并把第二项的负号提出来：
$$ dA_{i,a} = dQ_{i,a} - \frac{1}{N} dQ_{i,a} - \frac{1}{N} \sum_{k \neq a} dQ_{i,k} $$

观察后两项，它们都带有 $ -\frac{1}{N} $ 的因子，可以合并。把 $ -\frac{1}{N} dQ_{i,a} $ 放回求和符号里，让求和从 $ k=1 $ 覆盖到 $ N $：

$$ dA_{i,a} = dQ_{i,a} - \frac{1}{N} \left( dQ_{i,a} + \sum_{k \neq a} dQ_{i,k} \right) $$

括号里正好是该样本下所有动作 $ dQ $ 的**完整总和**：
$$ dA_{i,a} = dQ_{i,a} - \frac{1}{N} \sum_{k=1}^{N} dQ_{i,k} $$

写成矩阵形式，即：
$$ \boxed{dA = dQ - \text{mean}_{N}(dQ)} $$
这里的 $ \text{mean}_{N}(dQ) $ 就是对该样本所有动作梯度的平均值。

**为什么要减去均值？** 因为在中心化前向时，均值把各个动作耦合在了一起，反向传播时链式法则自然地把所有动作的平均梯度反哺回来，形成一个抵消机制。这就是“共模抑制”的数学来源。

假设有N个动作，Qi = V +Ai（i=1,2....N），未知数：1个V，N个动作就是N个未知数，N个方程。

未知数个数大于方程个数，因此有无穷多解。

例子：

3=a+b−e

4=a+c−e

5=a+d−e
​

e为b,c,d的均值，适当整理后，a=4（恒等，即Q的均值）

于是b-e=-1,c-e=0,d-e=1，无论b或者c,d如何取值，这些都是固定的。


实现了Q的解耦，比如需要Q整体抬高5，若调整A发现整体调高5，一加一减为0，于是只调整V，只有在某个动作产生影响很大时，才会调整A

A-Amean就是A内每个动作产生的回报相对于所有动作产生回报均值的偏差值，这些偏差值和一定为0，相当于减去了直流分量，高通滤波器，在实际网络中，是V+A =Q,解决唯一性是为了“告诉”网络应该如何调整，分配权重。


### 1.5 梯度经过“线性层”（$ V = X W_v $ 和 $ A = X W_a $）

现在，我们有了两个下游梯度：$ dV $（$ B \times 1 $）和 $ dA $（$ B \times N $）。它们要穿过各自的线性矩阵乘法，去更新底层的权重。

对于线性变换 $ Y = X W $，反向传播的链式法则是：

$$ \frac{\partial \mathcal{L}}{\partial W} = X^T \cdot \frac{\partial \mathcal{L}}{\partial Y} $$

**为什么会是 $ X^T $？** 因为矩阵乘法 $ Y_{i,j} = \sum_k X_{i,k} W_{k,j} $，当对 $ W_{k,j} $ 求偏导时，结果是 $ X_{i,k} $。将所有样本 $ i $ 汇总，就形成了矩阵乘法 $ X^T $ 乘以梯度矩阵。

应用到两个分支：

1.  **V 分支权重的最终梯度**：
    将之前得到的 $ dV = dQ $ 代入：
    $$ \frac{\partial \mathcal{L}}{\partial W_v} = X^T \cdot dQ \quad (D \times 1) $$

2.  **A 分支权重的最终梯度**：
    将第四步得到的 $ dA = dQ - \text{mean}(dQ) $ 代入：
    $$ \frac{\partial \mathcal{L}}{\partial W_a} = X^T \cdot (dQ - \text{mean}(dQ)) \quad (D \times N) $$


### 1.6 核心结论：两种权重的更新对比

设学习率为 $ \eta $，参数更新为 $ W_{new} = W_{old} - \eta \times \text{梯度} $。

- **更新 $ W_v $**：依靠 $ X^T \cdot dQ $。由于 $ dQ $ 保留了所有动作的完整误差，所以 $ W_v $ 会捕捉所有动作共同的、整体的变化趋势。
- **更新 $ W_a $**：依靠 $ X^T \cdot (dQ - \text{mean}(dQ)) $。

**关键区别在这里**：如果某一时刻，某个样本的所有动作误差都相等（例如 $ dQ = [0.5, 0.5] $），那么括号内的 $ dQ - \text{mean}(dQ) $ 会变成 $ [0.5, 0.5] - [0.5, 0.5] = [0, 0] $。这意味着 $ W_a $ 在这一步的更新量为 **0**。

只有当一个样本中不同动作的误差存在差异时（例如 $ dQ = [-0.8, 1.2] $），括号内的值才不为零，$ W_a $ 才会被更新，并专门朝着放大这种“动作间差异”的方向调整权重。

**最终总结**：整个过程没有引入任何人为判断，纯粹是链式法则对“中心化操作”求导带来的数学结果。加法节点导致了梯度的复制，中心化节点通过 $ k=a $ 和 $ k \neq a $ 两个系数的求和整理，导出了去均值化的梯度，而线性层的转置乘法决定了权重的最终调整方向。

## 五、改进三：N-step Learning —— 目标值看得更远

### 5.1 原理

标准 DQN 的目标值只用**一步**真实奖励，其余全靠目标网络"自举"（估计）：

$$\text{target} = r_t + \gamma\max_{a'}Q(s_{t+1},a')$$

N 步学习：用**连续 N 步的真实奖励**，自举部分只剩最后一步：

$$\text{target} = \underbrace{r_t + \gamma r_{t+1} + \cdots + \gamma^{N-1}r_{t+N-1}}_{G_t \;=\; \text{N 步累计折扣回报}} + \gamma^{N}\max_{a'}Q(s_{t+N},a')$$

直观理解：1 步目标只看眼前一步，容易"短视"；N 步目标看得更远，学习信号更接近真实回报。

### 5.2 偏差-方差折中

| | 真实奖励个数 | 自举部分 | 偏差 | 方差 |
|---|---|---|---|---|
| n=1 (TD) | 1 | $\gamma Q(s_{t+1})$ | 大 | 小 |
| n=N（折中） | N | $\gamma^N Q(s_{t+N})$ | 中 | 中 |
| n→∞ (MC) | 全部 | 0 | 小 | 大 |

n 越大越依赖真实数据（偏差小）、随机性越大（方差大）；取 N=3~5 是常用折中点。

### 5.3 代码修改（3 处）

1. **超参数**：新增 `N_STEP = 3`；
2. **ReplayBuffer**：从"存单步"改为"滚动窗口攒 N 步"，攒满才产出一条 $(s_0, a_0, G, s_N, done)$；
3. **update()**：折扣因子 `GAMMA` → `GAMMA ** N_STEP`（因为 rewards 已经是 G）。

### 5.4 经典坑：terminated vs truncated

目标里的 `(1-dones)` 必须用**真正终止**（terminated），不能用合并的 `done = terminated or truncated`：

- `terminated`（杆子倒了）→ 后面没有未来，未来项清零；
- `truncated`（撑满 500 步被掐断）→ 只是时间到了，未来价值**不该**清零。

所以训练循环里：传给 buffer 的是 `terminated`；`truncated` 时单独清空滚动窗口（避免跨回合拼接成假轨迹）。

## 六、改进四：Noisy Net —— 参数噪声代替 ε 探索

### 6.1 原理：从"动作噪声"到"权重噪声"

- **ε-贪婪**：以概率 ε 随机选一个动作——这是**动作空间里的噪声**，无论处于什么状态都以相同概率乱来；
- **Noisy Net**：在**网络权重里注入可学习的噪声**，每次前向都用"带噪声的权重"计算 → 输出带噪声 → `argmax` 偶尔翻车 = 探索。噪声大小**由网络自己学**（状态相关、自适应），不再需要手动调 ε。

### 6.2 数学：带噪声的线性层

普通线性层：$y = Wx + b$。Noisy 线性层把权重拆成"均值 + 噪声尺度 × 随机噪声"：

$$y = \underbrace{(\mu^W + \sigma^W \odot \varepsilon^W)}_{W} x + \underbrace{(\mu^b + \sigma^b \odot \varepsilon^b)}_{b}, \qquad \varepsilon \sim \mathcal{N}(0,1)$$

| 符号 | 是什么 | 作用 |
|---|---|---|
| $\mu$ | 可学习参数（均值） | 网络**真正要学**的权重 |
| $\sigma$ | 可学习参数（噪声尺度） | 网络**自己学**每个权重要加多少噪声 = 探索强度 |
| $\varepsilon$ | 每次前向**现采**的标准正态噪声 | 提供随机性（不参与梯度） |

关键：$W = \mu + \sigma\varepsilon$ 对 $\mu$、$\sigma$ **都可导** → 反向传播同时更新"权重"和"探索强度"（这就是重参数化技巧）。**测试时**不采样噪声、只用 $\mu$（`eval()` 模式 → 确定性推理）。

### 6.3 为什么比 ε 好

- ε 在任何状态都以相同概率乱来，即使在最优动作很明确的状态也会犯傻；
- Noisy Net 的探索强度是**状态相关的**：不确定的状态噪声大（探索），确定的状态噪声小（利用）；
- 不需要手动设计 ε 衰减曲线，**少一个超参数**。

### 6.4 代码修改

1. **新增 `NoisyLinear` 层**（核心）：训练时 `weight = μ + σ⊙ε`，测试时 `weight = μ`；
2. **网络所有 `nn.Linear` → `NoisyLinear`**（Dueling 结构保留）；
3. **删除 ε 的一切**：三个 `EPSILON_*` 超参数、`select_action` 的随机分支、`update()` 的 ε 衰减——**Noisy 与 ε 互斥**（有了参数噪声就不需要动作噪声，否则双重探索）。

## 七、改进五：Prioritized Replay —— 按误差加权抽样 + 存档续训

### 7.1 核心思想

普通回放池**均匀抽样**，把每条经验一视同仁。但每条经验的学习价值不同——**TD 误差大**的经验（网络预测与目标差得远）更能学到东西，"已经学会的"经验抽了也白抽。PER 的思路：**按 $|\text{TD误差}|$ 加权抽样**，让"该学的经验"更常被抽到：

$$p_i = |\delta_i| + \varepsilon$$

（$\varepsilon$ 是一个极小正数，防止优先级为 0 的经验永远抽不到。）

### 7.2 整体数据流（分四步）

**第一步：与环境交互 + N 步入队**

智能体由 Noisy Net 选动作（无 ε），与环境交互得到 $(s,a,r,s',\text{terminated},\text{truncated})$：

- `terminated=True`：任务真结束（失败/达成），后面没有未来；
- `truncated=True`：只是撑到时间上限被截断，未来价值**不该**清零。

把这一步先塞进 N 步滚动队列，分三种情况：

- 未满 N 步、未终止 → **什么都不做**，继续等；
- 攒满 N 步 → 产出一条经验，窗口**滑动**一步；
- `terminated` → 把窗口剩余**补成一条**，窗口清空；
- `truncated` → **清空窗口**（环境回合已断，不能跨回合拼接成假轨迹）。

（此时 `update()` 也会被调用，但前期经验不足 128 条，直接 `return` 跳过。）

**第二步：产出一条经验，进入求和树**

无论滑动还是终止，产出的都是同一种结构 $(s_0, a_0, G, s_N, done)$，其中 $G$ 是窗口内实际步数的折扣回报：

$$G = r_0 + \gamma r_1 + \cdots + \gamma^{N-1} r_{N-1}$$

两者的区别只在两点：**窗口内步数**（滑动用满 N 步、终止用实际步数）和 **done 标志**（滑动 `False`、终止 `True`）。

**第三步：入树（`_add` → `tree.add`）**

- 新经验用**当前最大优先级**入树（保证它至少被抽到一次、"接受审判"后再按真实误差调整）：

```python
self.tree.add(self.max_priority ** self.alpha, (s0, a0, G, sN, done))
```

- `add` 内部做了三件事：
  1. 算叶子下标 `idx = write + capacity - 1`；
  2. 把经验存进 `data[write]`（经验池下标）；
  3. 调 `update(idx, priority)`：把新优先级写进叶子，并**沿"叶子→父→…→根"这一条路径**把变化量向上传播（不是整棵树都改）；
  4. `write` 循环 +1（写满回绕覆盖最旧）、`size` 封顶 +1。

**第四步：采样 + 学习（`update()` 的核心）**

**(a) 采样 `sample(128)`**：

- 取根节点 `total`（= 池内**所有**优先级的和，不是 128）除以样本量，得段长 `segment = total / 128`；
- 把 $[0, total)$ 分成 128 段，每段**均匀取一个随机数** `value`，用 `get(value)` 从根向下走到叶子，返回（叶子下标, 优先级, 经验）；
- 万一命中"未写入的叶子"（data=None，防御①）：回退为从已写入的 `size` 条经验里**均匀随机取一条**；
- 循环 128 次，得到 128 组（下标, 优先级, 经验）。

**(b) 重要性采样权重**：

抽样概率与 IS 权重为：

$$P(i) = \frac{p_i^\alpha}{total}, \qquad w_i = \big(N \cdot P(i)\big)^{-\beta}, \qquad \text{再归一化 } w_i \leftarrow \frac{w_i}{\max_j w_j}$$

- $N = tree.size$（池内总条数）；`probs` 加极小下界防 `0^{-\beta}=\infty`（防御②），权重再 `nan_to_num` 兜底（防御③）；
- $\beta$ 从 0.4 **退火到 1.0**：前期允许偏差、激进地多学高优先级经验，后期趋于无偏。

**重要性采样系数**:

通用公式是目标分布除以当前采样分布。

理解，假设采样N次，在当前分布q(x)下被抽到了N·q(x)次，在目标分布p(x)下被抽到了N·p(x)次，为了让这几次抽样“伪装”成是从p(x)里抽出来的，我们需要给这N·q(x)个样本整体乘以一个倍数，使其总贡献等于N·p(x)。

即：N·q(x)·C  = N·p(x)，C就等于P(x)/q(x)

**(c) 计算损失并更新网络**：

$$q\_values = Q(s,a), \qquad \text{target} = G + \gamma^{N}\max_{a'}Q(s_{t+N},a')$$

$$\delta_i = q\_values_i - \text{target}_i, \qquad \text{loss} = \frac{1}{B}\sum_{i=1}^{B} w_i\, \delta_i^2$$

反向传播、梯度裁剪、`optimizer.step()` 更新权重。

**(d) 更新优先级**：用本批 TD 误差得到新优先级：

$$p_i^{\text{new}} = |\delta_i| + \varepsilon \qquad(\text{NaN/inf 兜底，防御④})$$

对采样的 128 个叶子下标逐一调用 `tree.update`，把新优先级沿"叶子→根"路径传播；同时维护 `max_priority = max(max_priority, rp)`（初值 1.0，故最低为 1.0）。

### 7.3 存档 / 续训（checkpoint）

- **自动存档**：每 `SAVE_INTERVAL = 50` 回合保存一次完整状态，包含：
  - 两个网络权重（`q_net` / `target_net`）；
  - 优化器状态（Adam 动量，不存则续训"断档"）；
  - 经验池内部（求和树数据、N 步窗口、$\beta$、`max_priority`）；
  - 已完成的回合数 + 历史奖励（续训后曲线连续）。
- **暂停**：训练中按 `Ctrl+C` → 先保存再退出；
- **续训**：下次启动 `load_checkpoint` 检测到存档，从断点回合继续（打印"从第 X 回合继续"）。

### 7.4 本轮踩过的坑（数值防御）

1. `get` 命中未写入叶子 → 回退随机有效样本，防止 `None` 混进 batch；
2. `probs` 加下界，防 `0^{-\beta}=\infty`；
3. IS 权重 `nan_to_num` 兜底，防 inf/nan 污染损失；
4. 优先级 `nan_to_num`，防 NaN 进入求和树——否则 `value <= NaN` 恒为 False，抽样会"一路往右"走到未写入叶子，直接崩溃。

直白的的整体理解（可能有点小错误）：

首先DQN智能体循环与环境交互（由 Noisy Net 选择动作），得到了s,a,r,ns,done(中止状态，分为truncated与terminated ,前者代表回合结束被接断，后者代表失败，中止等），存入n-step的队列，若不满足terminated或n步的条件，什么都不做，如果truncated，则清空n-step队列所有数据。然后继续用update更新网络，实际上由于前期不足128个样本，这一步也跳过，状态转移后，再与环境交互。直到存满了n-step或者terminated，把这个队列内的最开始的(s,a)状态对，和这个队列内实际获得的G，与下一步的sN,以及done状态作为数据进入树结构（经验池）（满了n-step就是滑动，terminated就是清空，二者还有队列内步数的区别，done的区别，别的都一致，都是s0,a0,G,sN,done的结构）。

这个数据进入树结构用的是_add（当然实际上是实例tree的add方法），add给这个数据最大的priority（self.max_priority ** self.alpha,pi的alpha次方，只不过这个pi是最大的），然后用idx = self.write + self.capacity - 1 ，获取其树结构下标，再给这个经验编了一个经验池下标（self.data[self.write] = data），再用update更新整个树结构的所有节点优先级，write,size分别+1。这就算一个有效经验。
在经验池未满128个经验前，都是循环做上述，没有别的操作（存档续训先不管）。

一旦满足了128个经验，agent.update()就要发力了，先用sample函数进行采样：total(指的是经验池内所有优先级的数据，并非是128个，就是根的数据）/样本量，得到了128个数据（128个区间），对每个区间（数据，样本量），均匀选取一个随机数，用get函数得到一个叶子，返回这个叶子，包含叶子树结构下标，具体优先级分数，具体经验，若遇到none，则进行防御：从已写入的size个经验的经验池内，随机选一个经验池下标，转化为叶子下标，调用tree的公共属性获取其具体优先级分数，具体经验。叶子下标，优先级分数，数据进入一个列表，重复128次（样本次数),然后对于优先级分数转化为数组，计算抽样概率

np.array(tree_values) / self.tree.total，这边给一个极小下届防止错误。然后计算重要性采样修正偏差，
weights = (self.tree.size * probs) ** (-self.beta)，self.tree.size 是经验池所有的样本数量，prob就是这个128个样本各自的优先级分数数组，self.beta是退火，前期允许偏差积极学习，后续逐渐到1，实现无偏，然后就是偏差系数的兜底，与数据解包，返回sarnsd，这个r指的是多步的G，叶子下标和重要性修正系数。当然这只是update的第一步，总结来说就是采样得到了样本数量的sarnsd以及重要性修正系数（转为数组），利用样本数量的sarnsd计算当前Q和目标Q，得到了td_error，用loss = (weights * td_errors ** 2).mean()计算loss,然后反向传播，更新网络权重。切断梯度追踪，用raw_priorities = td_errors.detach().squeeze(1).abs().cpu().numpy() + PER_EPS得到新的优先级Pi（同样兜底），然后调用树的update_priorities针对indices（也被zip为元组）（样本数量个） raw_priorities用树的update进行更新128次（样本数量次），每次更新都会改变整个树结构（即从indices内某个叶子坐标开始向上更新到根）。这边循环内每一步都self.max_priority = max(self.max_priority, rp)，获得这个128个样本内最大的pi，或者都小于1时，选择1。到这边智能体的update算完了，进行状态转移与循环，回合内进行网络复制与β退火，不算是核心逻辑。这个理解对吗